# 📊 EU Stock Market Forecasting
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/eu-stock-forecasting/training/notebook.ipynb)

Forecast the four big European stock indices of the 1990s (**DAX, SMI, CAC, FTSE**; R's classic
`EuStockMarkets` dataset, 1,860 business days 1991-1998). Five forecasters are compared on every
index: **ARIMA** (order by AIC), **Holt-Winters** (additive trend), **VAR** (all four indices, lag by AIC),
a **Keras MLP** and a **Keras LSTM**, plus the naive "tomorrow = today" baseline. Per index, the classical
model with the fewest parameters among those within 2% of the best validation error is deployed; the [Space](https://huggingface.co/spaces/shalev396/eu-stock-forecasting)
refits it on the history up to any cutoff date and forecasts the following business days.

All logic lives in `src/` and `../model/model.py` (the forecasters the Space runs); this notebook calls it step by step.

**Contents:** 1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "eu-stock-forecasting"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has; nothing platform-specific is installed or special-cased.
import tensorflow as tf
device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print("device:", device)

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from src import data_setup, engine, export, model_builder, utils   # importing src puts ../model on sys.path
from src.config import Config
import model as M                                                    # ../model/model.py

print(utils.lib_versions("statsmodels", "pandas", "numpy", "sklearn", "tensorflow", "keras"))

## 2. Config
Every hyperparameter lives in `src/config.py`: a 100-day validation window and a 100-day test window
at the end of the data, the ARIMA grid, the VAR lag limit, the neural-net sizes (20-day window, MLP 64-32,
LSTM 64 units) and the multi-step backtest. `SMOKE_TEST=1` keeps the last 400 rows, shrinks every grid
and model, trains the nets for 1 epoch on the DAX only and exports to `outputs/smoke/model` instead of `../model`.
A full run takes ~4 min on a desktop CPU (up to ~17 min when it is busy; mostly the ARIMA grids and the Keras nets).

In [ ]:
os.environ.setdefault("SMOKE_TEST", "0")   # "1" = quick smoke run; a SMOKE_TEST set outside the notebook wins
PUSH_TO_HUB = False                         # upload the exported model folder to the Hub (section 8)

cfg = Config()
utils.set_seeds(cfg.seed, "tensorflow")
print("export to:", cfg.model_dir)
cfg

## 3. Data
`EuStockMarkets` comes from R's `datasets` package through `statsmodels.datasets.get_rdataset`
(fallback: the Rdatasets CSV mirror) and is cached in `training/data/`. The R series has no calendar
(start 1991.496, 260 obs/year), so rows get a **synthetic business-day index starting 1991-07-01**
(weekdays, no holidays): the dates are approximate, the order and spacing are real.

Split, chronological and never shuffled: **fit** (everything before) · **validation** (100 days: picks the
deployed method, early-stops the nets) · **test** (last 100 days, reported once).

In [ ]:
frame = data_setup.load_eustocks()
splits = data_setup.make_splits(frame, cfg)
display(frame.describe().round(1))
splits.summary()

In [ ]:
display(Image(filename=str(export.plot_series(splits, cfg.outputs_dir / "series.png"))))

Stationarity (the classical workflow): the ADF test cannot reject a unit root in the **levels**
(p > 0.05) while the **first differences** are clearly stationary, so ARIMA uses d = 1.

In [ ]:
data_setup.adf_table(splits.fit).round(4)

## 4. Load model
The deployable forecasters are functions in `../model/model.py` (`fit_holt_winters`, `fit_arima` /
`filter_arima`, `fit_var`, `forecast`). The two neural nets are training-only baselines from
`src/model_builder.py`.

In [ ]:
for name, description in model_builder.CLASSICAL.items():
    print(f"{name:13s} {description}")
nets_preview = {name: build(cfg) for name, build in model_builder.BUILDERS.items()}
for name, net in nets_preview.items():
    print(f"{name}: {utils.count_params(net):,} parameters")
nets_preview["LSTM"].summary()

## 5. Training
Everything is fit on the **fit window** only. Per index: an ARIMA(p,1,q) grid (p, q ≤ 3) ranked by AIC,
Holt-Winters with additive trend, the MLP and the LSTM (Adam + MSE on MinMax-scaled closes, early
stopping on validation). VAR is fit once on all four indices.

In [ ]:
t0 = time.perf_counter()
var_fit = engine.fit_var(splits, cfg)
fits = {ix: engine.fit_classical(splits, ix, cfg) for ix in cfg.indices}
fits[cfg.target]["aic_table"].head(8)

In [ ]:
nn_data, nets, histories = {}, {}, {}
for ix in cfg.nn_indices:                # smoke: the target index only
    nn_data[ix] = data_setup.prepare_nn_data(splits, ix, cfg)
    nets[ix], histories[ix] = {}, {}
    for name, build in model_builder.BUILDERS.items():
        utils.set_seeds(cfg.seed, "tensorflow")
        nets[ix][name] = build(cfg)
        histories[ix][name] = engine.fit_nn(nets[ix][name], nn_data[ix], cfg)
        print(f"[{ix}] {name}: {len(histories[ix][name]['loss'])} epochs, "
              f"best val loss {min(histories[ix][name]['val_loss']):.5f}")
train_time_s = time.perf_counter() - t0
print(f"training: {train_time_s:.0f}s")
display(Image(filename=str(export.plot_training_curves(histories[cfg.target], cfg.target,
                                                       cfg.outputs_dir / "training_curves.png"))))

## 6. Evaluation
**One-step-ahead walk-forward** over validation + test: parameters stay frozen, and each forecast sees the
actual closes up to the day before (the setting of R's `forecast::accuracy()`, whose table is reproduced:
ME, RMSE, MAE, MPE, MAPE, MASE). **Selection rule** (fixed before the test window is looked at, and it never reads it): rank the classical
models (ARIMA, Holt-Winters, VAR) by **validation** MAPE; among all models within **2% (relative)** of the
best one, deploy the one with the **fewest fitted parameters** (ties go to the lower validation MAPE). The
differences between the classical models are tiny, so a 164-parameter VAR(10) only wins when no small model
comes close. Test numbers are only reported.

In [ ]:
preds = {ix: engine.one_step_predictions(splits, ix, fits[ix], var_fit, nets.get(ix), nn_data.get(ix), cfg)
         for ix in cfg.indices}
tables = {ix: engine.evaluate(splits, ix, preds[ix]) for ix in cfg.indices}
n_params = {ix: engine.classical_param_counts(tables[ix], fits[ix], var_fit[1], cfg) for ix in cfg.indices}
deployed = {ix: engine.select_deployed(tables[ix], n_params[ix], cfg.select_tolerance) for ix in cfg.indices}
print("fitted parameters:", n_params[cfg.target])
print("deployed:", deployed)
tables[cfg.target].round(3)

In [ ]:
# test MAPE of every variant kind on every index (the ARIMA order / VAR lag differ per index)
pd.DataFrame({ix: t["MAPE"].rename(index=lambda n: n.split("(")[0]) for ix, t in tables.items()}).round(3)

In [ ]:
display(Image(filename=str(export.plot_comparison(tables, deployed, cfg.outputs_dir / "model_comparison.png"))))
display(Image(filename=str(export.plot_overlay(splits, cfg.target, preds[cfg.target], deployed[cfg.target],
                                               cfg.outputs_dir / "holdout_overlay.png"))))

**Multi-step backtest** — exactly what the Space serves: from origins every 10 days of the test window,
`model.forecast` sees only the history up to the origin and predicts the next 20 business days
(Holt-Winters and VAR are refit, ARIMA re-filters with its fixed coefficients). MAE in index points.

In [ ]:
specs = {ix: engine.serving_spec(fits[ix], var_fit[1], deployed[ix], cfg) for ix in cfg.indices}
backtests = {ix: engine.backtest(splits, ix, specs[ix], cfg) for ix in cfg.indices}
display(pd.DataFrame({ix: b["table"]["MAE"] for ix, b in backtests.items()}).round(1).rename_axis(
    f"{cfg.backtest_horizon}-day MAE"))
display(Image(filename=str(export.plot_backtest(splits, cfg.target, backtests[cfg.target], specs[cfg.target],
                                                cfg.outputs_dir / "forecast_backtest.png"))))

## 7. Inference
The deployable model is just `config.json` (method + orders + ARIMA coefficients per index) and
`eustocks.csv` (the data it forecasts from). `export.save_model` writes both to `cfg.model_dir`; then
`model.load(dir).predict(cutoff, horizon, index)` runs exactly what the Space and the endpoint run, on the
requests in `../space/examples/requests.json`.

In [ ]:
export.save_model(cfg, frame, specs)
predictor = M.load(cfg.model_dir, "cpu")
requests = utils.load_json(utils.SPACE_DIR / "examples" / "requests.json")
for cutoff, horizon, index in requests:
    start = time.perf_counter()
    out = predictor.predict(cutoff, horizon, index)
    print(f"{index} after {cutoff} (+{horizon}d) {out['model']}: MAE {out['mae']} vs naive {out['naiveMae']} "
          f"({time.perf_counter() - start:.2f}s)")

In [ ]:
cutoff, horizon, index = requests[0]
out = predictor.predict(cutoff, horizon, index)
fig, ax = plt.subplots(figsize=(9, 3.5))
for key, style in (("history", "-"), ("actual", "-"), ("forecast", "-"), ("naive", "--")):
    part = pd.DataFrame(out[key])
    if len(part):
        ax.plot(pd.to_datetime(part["date"]), part["price"], style, label=key)
ax.set_title(f"{index} from {cutoff}: {out['model']}")
ax.legend()
plt.show()

In [ ]:
byo = sorted((utils.SPACE_DIR / "examples").glob("*.csv"))[0]
out = predictor.predict_series(byo, 20)
print(byo.name, out["model"], out["forecast"][:3])

## 8. Export
`export.export` writes `metrics.json` (the deployed DAX model + the comparison of every variant on every
index + the backtest + the selection rule: fewest fitted parameters within 2% of the best validation MAPE,
test metrics never used), the card plots in `assets/`, and refreshes the model card (`README.md`). **With
`SMOKE_TEST=0` this overwrites `../model`.** To publish, set `PUSH_TO_HUB = True` in section 2. The token
comes from `hf auth login` (local) or an `HF_TOKEN` secret / environment variable (Colab: the Secrets panel).
Smoke runs are never uploaded.

In [ ]:
params = dict(n_params[cfg.target])
params |= {name: utils.count_params(net) for name, net in nets[cfg.target].items()}
metrics = export.export(cfg, splits=splits, tables=tables, deployed=deployed, specs=specs, backtests=backtests,
                        params=params, preds=preds, histories=histories, device=device.lower(),
                        train_time_s=train_time_s)
display(pd.Series(metrics["metrics"]).round(4))

for path in sorted(cfg.model_dir.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(f"{path.relative_to(cfg.model_dir).as_posix():32s} {path.stat().st_size / 1e3:8.1f} KB")

if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))